# Kaggle 50 Startups Profit Prediction
## Section 1：Business Understanding

### 1. 商業問題
How can we predict a startup company’s profit based on its spending in different business areas?

### 2. 為什麼要預測 startup profit
新創公司通常資源有限，管理者需要知道資金應該投入在哪些地方，才能提高公司利潤。

### 3. 資料中的支出欄位代表什麼商業意義
- **R&D Spend**：研發支出，是創新的核心。
- **Administration**：行政管理支出，反映營運規模與固定成本。
- **Marketing Spend**：行銷支出，是拓展市場的投資。
- **State**：所在州別，可能影響稅收、人才與市場機會。

### 4. 為什麼這是一個 Regression Problem
因為我們的目標變數 `Profit` 是連續數值 (Continuous Numeric Variable)，而非類別，所以屬於監督式學習的迴歸問題。

### 5. 專案成功標準
1. 預測新創公司的 Profit。
2. 找出影響 Profit 的重要因素。
3. 協助管理者做預算分配。
4. 提供投資者或商業分析師參考。
5. 建立可解釋的機器學習模型。

## Section 2：Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 讀取資料
df = pd.read_csv('../data/50_Startups.csv')

# 基本觀察
display(df.head())
display(df.tail())
df.info()
display(df.describe())
print('Missing Values:', df.isnull().sum())
print('Duplicated Values:', df.duplicated().sum())
print(df['State'].value_counts())

In [ ]:
# 圖表分析
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.scatterplot(x='R&D Spend', y='Profit', data=df, ax=axes[0,0]).set_title('R&D vs Profit')
sns.scatterplot(x='Marketing Spend', y='Profit', data=df, ax=axes[0,1]).set_title('Marketing vs Profit')
sns.scatterplot(x='Administration', y='Profit', data=df, ax=axes[1,0]).set_title('Administration vs Profit')
sns.boxplot(x='State', y='Profit', data=df, ax=axes[1,1]).set_title('State vs Profit')
plt.tight_layout()
plt.show()

# Correlation
plt.figure(figsize=(6, 4))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Profit Distribution
sns.histplot(df['Profit'], bins=15, kde=True)
plt.title('Profit Distribution')
plt.show()

### 分析說明
1. **R&D Spend** 看起來與 Profit 有非常明顯的正相關。
2. **Marketing Spend** 也與 Profit 有關，但相關性較 R&D 弱。
3. **Administration** 的影響較為分散，相關性最弱。
4. **State** 為類別資料，需要做 One-Hot Encoding。

| 特徵 | 專家定位 | 預期重要性 | 建議 |
|---|---|---|---|
| R&D Spend | 核心成長因子 | 很高 | 一定保留 |
| Marketing Spend | 市場擴張因子 | 中高 | 保留，但注意共線性 |
| Administration | 營運成本／規模因子 | 低到中 | 先保留，後續評估 |
| State | 地區輔助因子 | 低到中 | One-Hot 後保留，謹慎解讀 |

## Section 3：Data Preprocessing

本階段需要將原始資料轉換為模型可用格式：
- 分離 X (Features) 與 y (Target)。
- 將資料集切割為 Training Set 與 Testing Set 以利後續驗證模型的泛化能力。
- 針對類別變數 `State` 的 One-Hot Encoding，我們將在 Modeling 階段交給 Scikit-Learn 的 `Pipeline` 與 `ColumnTransformer` 來自動處理，這是一種更嚴謹且易於部署的做法。

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Profit', axis=1)
y = df['Profit']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)
print('Train Shape:', X_train.shape)
print('Test Shape:', X_test.shape)

## Section 4：Modeling

本專案改用 `RandomForestRegressor`，並結合 `sklearn.pipeline.Pipeline` 與 `ColumnTransformer`，這能同時幫助我們優雅地處理類別變數 (One-Hot Encoding) 以及捕捉資料中的非線性關係。

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import os

# 1. 建立前處理器
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), ['State'])
    ],
    remainder='passthrough'
)

# 2. 建立 Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 3. 訓練模型
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 檢視 Pipeline 模型
display(model)

# (附註：Random Forest 使用 feature_importances_ 而非 coef_)

## Section 5：Evaluation

### 指標意義
- **MAE** (平均絕對誤差)：越低越好。
- **MSE** (平均平方誤差)：越低越好。
- **RMSE** (均方根誤差)：越低越好。若遠大於 MAE，代表有少數極端誤差。
- **R² Score**：模型解釋力，越接近 1 越好。

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

evaluation_result = pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'RMSE', 'R2 Score'],
    'Value': [mae, mse, rmse, r2]
})
display(evaluation_result)
evaluation_result.to_csv('../outputs/evaluation_metrics.csv', index=False)

comparison = pd.DataFrame({
    'Actual Profit': y_test,
    'Predicted Profit': y_pred,
    'Error': y_test - y_pred
})
comparison.to_csv('../outputs/actual_vs_predicted.csv', index=False)

plt.figure(figsize=(8,5))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Profit')
plt.ylabel('Predicted Profit')
plt.title('Actual vs Predicted')
plt.show()

## Section 6：Deployment
將訓練好的模型透過 joblib 儲存起來，準備讓 Streamlit 與 FastAPI 進行應用部署。